In [1]:
from dotenv import load_dotenv    
import os                         

load_dotenv()                    

from langchain_openai import ChatOpenAI                          
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated                          


class State(TypedDict):
    messages: Annotated[list, add_messages]   

In [2]:

llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)


def chatbot(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    
    
    response = llm.invoke([system] + state["messages"])
    
    
    return {"messages": [response]}

In [3]:

graph_builder = StateGraph(State)              

graph_builder.add_node("chatbot", chatbot)     

graph_builder.add_edge(START, "chatbot")       
graph_builder.add_edge("chatbot", END)        

graph = graph_builder.compile()

In [5]:

result = graph.invoke({
    "messages": [HumanMessage(content="Who is the president of Nigeria?")]
})


for msg in result["messages"]:
    if hasattr(msg, 'content'):
        
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"{role}: {msg.content}\n")

Human: Who is the president of Nigeria?

AI: As of October 2023, the president of Nigeria is Bola Ahmed Tinubu. He took office on May 29, 2023. Please verify this information from a reliable source, as political situations can change.

